# LoRA: run OK check, smoke run, LR sweep, final runs

One notebook for both LoRA ranks. **Eddie: `RANK = 4`. Dylan: `RANK = 32`** (cell 3). Everything else is the same.

Order (Paddy's 14 Oct list):
1. Setup (T4, Drive, latest code, images into `data/raw`)
2. `python -m src.run` self-check, must end with **run OK**
3. Settings
4. Smoke run, timed. It *is* the first sweep run (lr 1e-3, seed 0), so it isn't wasted
5. LR sweep, validation only: seeds 0–1 first (the minimum), then 2–4 if time allows
6. Summary: pick the lowest mean val CE, flag grid edge / 50-epoch cap, compare seed 2 with frozen, print a decision-log draft
7. Final runs (seeds 0–4) — only after the chosen lr is committed in `configs/r<RANK>.yaml`

Sweep runs go to `runs/pilot/lora_lr/` and `results/pilot_lora_lr_r<RANK>.csv` on Drive, never `runs/` or `runs.csv`.
The test set is never loaded.

**If Colab disconnects:** rerun cells 1 and 3, then the same cell you were on. Finished runs are skipped, a half-done run resumes from `last.pt`.

In [ ]:
# 1. Setup (~3-5 min). First: Runtime -> Change runtime type -> T4 GPU
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/aai-project"
REPO = "/content/Advanced-AI"

# the repo is private: put a GitHub token in Colab Secrets (key icon, left bar) as GH_TOKEN, notebook access on
if not os.path.exists(f"{REPO}/.git"):   # .git, not just the folder: a failed clone can leave an empty folder
    !rm -rf {REPO}
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN").strip()  # a pasted secret can carry a trailing newline
    !git clone -q https://{TOKEN}@github.com/dylan-teehan-skc/Advanced-AI.git {REPO}
assert os.path.exists(f"{REPO}/.git"), "clone failed: check the GH_TOKEN secret"
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q -r requirements.txt

# the self-check and run() read images from data/raw/EuroSAT_RGB inside the repo (Paddy's note)
if not os.path.exists("data/raw/EuroSAT_RGB"):
    !cp "{DRIVE}/EuroSAT_RGB.zip" /content/
    !unzip -q /content/EuroSAT_RGB.zip -d data/raw
!ls data/raw/EuroSAT_RGB
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# 2. Self-check (~2-4 min, CPU, downloads the ViT once). Last line must be: run OK
!python -m src.run

In [ ]:
# 3. Settings. Rerun this after any disconnect.
import time
from pathlib import Path

import pandas as pd
import torch
import yaml

from src.run import load_config, run

RANK = 4                      # Eddie 4, Dylan 32
COND = f"r{RANK}"
LRS = [1e-4, 3e-4, 1e-3]      # Paddy's grid
SWEEP_ROOT = Path(DRIVE) / "runs/pilot/lora_lr"
SWEEP_CSV = f"{DRIVE}/results/pilot_lora_lr_{COND}.csv"
FINAL_CSV = f"{DRIVE}/results/runs_{COND}.csv"   # own CSV per rank, so two accounts never append to one file

base = load_config(f"configs/{COND}.yaml")
assert base["n_per_class"] == 20, "old config: git pull didn't bring in Paddy's N = 20 commit"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")


def lr_folder(lr):
    return SWEEP_ROOT / f"{COND}_lr{lr:g}"


def sweep_config(lr):
    """configs/r<RANK>.yaml with only lr changed, in its own folder so runs at different lrs never collide."""
    folder = lr_folder(lr)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "config.yaml").write_text(yaml.safe_dump({**base, "lr": lr}))
    return folder / "config.yaml", folder


def epochs_run(folder, seed):
    """How many epochs the run actually trained (early stop or the 50 cap), from its last checkpoint."""
    return torch.load(Path(folder) / f"{COND}_s{seed}" / "last.pt", map_location="cpu")["state"]["epoch"]


# a leftover folder in runs/ would be skipped (row.json) or resumed (last.pt) instead of run fresh
stale = sorted(p.name for p in (Path(DRIVE) / "runs").glob(f"{COND}_s*"))
print(f"Already in runs/: {stale}. Fine if they are your final runs resuming; otherwise move them out before cell 7"
      if stale else "runs/ is clear for the final runs")
old = sorted(str(p.relative_to(SWEEP_ROOT)) for p in SWEEP_ROOT.glob(f"{COND}_lr*/{COND}_s*"))
print(f"sweep runs already on Drive (will be skipped or resumed): {old}" if old else "no sweep runs yet")

In [ ]:
# 4. Smoke run on split_v1, timed. Same settings as sweep run lr 1e-3 / seed 0, so it counts towards the sweep.
cfg, folder = sweep_config(1e-3)
row = run(cfg, seed=0, out_root=folder, results_csv=SWEEP_CSV)
ep = epochs_run(folder, 0)
mins = row["train_time_s"] / 60
print(f"\nsmoke run: {mins:.1f} min training, {ep} epochs ({row['train_time_s'] / ep:.0f} s/epoch), "
      f"best epoch {row['best_epoch']}, val CE {row['val_ce']:.4f}, balanced acc {row['val_bal_acc']:.3f}, "
      f"{row['trainable_params']:,} trainable")
print(f"rough budget if every run is like this one (+~1 min each to load and score):\n"
      f"  full sweep (14 more) + 5 finals: {19 * (mins + 1) / 60:.1f} h\n"
      f"  seeds 0-1 sweep (5 more) + 5 finals: {10 * (mins + 1) / 60:.1f} h")
print("Lower lrs usually run more epochs, so treat this as a floor.")

In [ ]:
# 5a. Sweep, seeds 0-1 for every lr (the minimum). One seed at a time across all lrs,
#     so if Colab cuts us off, every lr has the same seeds.
def sweep(seeds):
    for seed in seeds:
        for lr in LRS:
            cfg, folder = sweep_config(lr)
            r = run(cfg, seed=seed, out_root=folder, results_csv=SWEEP_CSV)
            print(f"== lr {lr:g} seed {seed}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, "
                  f"best epoch {r['best_epoch']} of {epochs_run(folder, seed)}, {r['train_time_s'] / 60:.1f} min\n")


sweep([0, 1])

In [ ]:
# 5b. Seeds 2-4, to match frozen's 5 seeds per value. Skip if time is short, and say so in the log.
sweep([2, 3, 4])

In [ ]:
# 6. Summary and pick. Rule: lowest mean validation CE, over the seeds every lr has finished.
df = pd.read_csv(SWEEP_CSV)
df = df[df["condition"] == COND].drop_duplicates(["lr", "seed"], keep="last")
seeds = sorted(set.intersection(*(set(g["seed"]) for _, g in df.groupby("lr"))))
missing = set(LRS) - set(df["lr"])
assert not missing, f"no finished runs yet for lr {missing}"
df = df[df["seed"].isin(seeds)].copy()
df["epochs"] = [epochs_run(lr_folder(lr), s) for lr, s in zip(df["lr"], df["seed"])]

t = df.groupby("lr").agg(runs=("seed", "count"), ce=("val_ce", "mean"), ce_sd=("val_ce", "std"),
                         acc=("val_bal_acc", "mean"), acc_sd=("val_bal_acc", "std"),
                         best_epochs=("best_epoch", list), epochs=("epochs", list),
                         mins=("train_time_s", lambda s: round(s.mean() / 60, 1))).reset_index()
print(f"{COND}, N = 20, seeds {seeds}\n")
print(t.round(4).to_string(index=False))

best = t.loc[t["ce"].idxmin()]
CHOSEN_LR = float(best["lr"])
print(f"\nRule: lowest mean val CE -> lr = {CHOSEN_LR:g}")
flags = []
if CHOSEN_LR in (min(LRS), max(LRS)):
    flags.append(f"the best lr ({CHOSEN_LR:g}) is at the edge of the grid; the true best may lie outside it")
cap = base["max_epochs"]
hit = df[df["epochs"] >= cap]
if len(hit):
    flags.append("hit the 50-epoch cap: " + ", ".join(f"lr {r.lr:g} s{r.seed} (best epoch {r.best_epoch})"
                                                       for r in hit.itertuples()))
if len(seeds) < 5:
    flags.append(f"only seeds {seeds} for every lr (frozen used 5)")
for f in flags:
    print("FLAG:", f)

# seed 2 was frozen's weak seed (0.605); is it weak for LoRA too?
frozen = pd.read_csv("results/runs.csv").query("condition == 'frozen'").set_index("seed")["val_ce"]
mine = df[df["lr"] == CHOSEN_LR].set_index("seed")["val_ce"]
print("\nval CE per seed, frozen vs this rank at the chosen lr:")
print(pd.DataFrame({"frozen": frozen, f"{COND} lr {CHOSEN_LR:g}": mine}).round(3).to_string())

# draft decision-log entry (Paddy's format). Check it, edit the Why if needed, then paste into your log.
per_lr = "; ".join(f"lr {r.lr:g} gave CE {r.ce:.3f} +/- {r.ce_sd:.3f}" for r in t.itertuples())
print(f"""
---- draft for docs/decision-log ----
## {time.strftime('%-d %b %Y')} - {COND} learning rate = {CHOSEN_LR:g}
- **Decision:** {COND} uses lr = {CHOSEN_LR:g}, chosen by the sweep rule: lowest mean validation CE at N = 20 over 1e-4, 3e-4, 1e-3.
- **Why:** {len(seeds)} seeds each: {per_lr}.{' ' + '. '.join(f[0].upper() + f[1:] for f in flags) + '.' if flags else ' The best is inside the grid and no run hit the 50-epoch cap.'} Results in `results/pilot_lora_lr_{COND}.csv`.
- **Affects:** `lr` in `configs/{COND}.yaml`, the final {COND} runs.""")

### Before cell 7 (on your laptop, ~5 min)
1. `git pull`
2. In `configs/r<RANK>.yaml` set `lr:` to the chosen value **written with a dot and a signed exponent**, e.g. `3.0e-4`. PyYAML reads `3e-4` as a *string*, not a number.
3. Paste the decision-log draft into your log, copy `pilot_lora_lr_r<RANK>.csv` from Drive `aai-project/results/` into the repo's `results/`.
4. Commit and push. Then run cell 7 (it pulls first, so each row records the commit with the chosen lr).

In [ ]:
# 7. Final runs: seeds 0-4 at the chosen lr into runs/ and runs_r<RANK>.csv. Validation only.
!git pull -q
!git log --oneline -1
CHOSEN_LR = globals().get("CHOSEN_LR")  # after a restart, type it instead, e.g. CHOSEN_LR = 3e-4
assert CHOSEN_LR is not None, "set CHOSEN_LR (cell 6 sets it, or type it)"
cfg = load_config(f"configs/{COND}.yaml")
assert isinstance(cfg["lr"], float), f"lr in configs/{COND}.yaml read as {cfg['lr']!r}; write it like 3.0e-4"
assert abs(cfg["lr"] - CHOSEN_LR) < 1e-12, f"configs/{COND}.yaml still has lr {cfg['lr']:g}; commit the chosen lr and push first"

for seed in range(5):
    r = run(f"configs/{COND}.yaml", seed=seed, out_root=f"{DRIVE}/runs", results_csv=FINAL_CSV)
    print(f"== {r['run_id']}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, best epoch {r['best_epoch']}\n")

final = pd.read_csv(FINAL_CSV)
print(final[["run_id", "lr", "best_epoch", "val_ce", "val_bal_acc", "train_time_s", "gpu", "git_commit"]].to_string(index=False))
print(f"\nmean val CE {final['val_ce'].mean():.3f} +/- {final['val_ce'].std():.3f}, "
      f"balanced acc {final['val_bal_acc'].mean():.1%}   (frozen: 0.559 +/- 0.026, 80.2%)")